# 4 — Analysis

Reproduces every results table of the dissertation — Chapter 5 (Tables 5.1–5.16) and
Appendix A (Tables A.1–A.3) — from the experiment artifacts
(`artifacts_thesis_experiment_trainvaltest/tables/`) and the aligned panel.

Each table is displayed inline in the same format as the thesis.

## Setup — inputs and helpers

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 60)

# All paths are anchored to the project folder explicitly.
BASE = Path("/Users/nunovieira/Masters/Thesis Final")
PANEL_CSV = BASE / "data" / "processed" / "aligned_panel_20220101_20260430.csv"
ART_TABLES = BASE / "artifacts_thesis_experiment_trainvaltest" / "tables"

ASSETS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT"]
SHORT = {"BTCUSDT": "BTC", "ETHUSDT": "ETH", "SOLUSDT": "SOL", "XRPUSDT": "XRP", "ALL": "ALL"}
ALGOS = ["PPO", "A2C", "RecurrentPPO"]

# Evaluation window boundaries (match StudyConfig in 3_Experiment).
VAL_END, DATA_END = "2025-08-31 23:00", "2026-04-30 16:00"
ANNUALIZATION = 8760  # hourly bars -> sqrt(8760) annualisation

# ---- sweep artifacts: one row per run = (cell_id, algorithm, target, seed) ----
summary = pd.read_csv(ART_TABLES / "summary_ablation.csv")
assert len(summary) == 1200, "expected exactly 1,200 final runs"
# target ALL = portfolio agent; a single symbol = asset-specific agent
summary["arch"] = np.where(summary["target"] == "ALL", "portfolio", "asset")

# ---- aligned panel: close prices + funding events for the baseline ----
panel = pd.read_csv(PANEL_CSV, parse_dates=["time"])
close = panel.pivot(index="time", columns="symbol", values="futures_close")[ASSETS]
fund = panel.pivot(index="time", columns="symbol", values="funding_rate_event")[ASSETS].fillna(0.0)
test_close = close.loc[(close.index > VAL_END) & (close.index <= DATA_END)]
test_fund = fund.reindex(test_close.index).fillna(0.0)

print(f"runs: {len(summary):,} | test window: {test_close.index.min()} -> "
      f"{test_close.index.max()} ({len(test_close):,} hourly bars)")


def pm(m, s, dm=2, ds=None):
    """'mean ± std' string."""
    ds = dm if ds is None else ds
    return f"{m:.{dm}f} ± {s:.{ds}f}"


def n_pct(n, total):
    return f"{n:,} ({n / total:.1%})"


def f3(v):
    """3-decimal string with half-up rounding via the 4-decimal artifact value
    (the same two-step lineage the thesis tables were produced with)."""
    from decimal import Decimal, ROUND_HALF_UP
    d4 = Decimal(str(float(v))).quantize(Decimal("0.0001"), rounding=ROUND_HALF_UP)
    return str(d4.quantize(Decimal("0.001"), rounding=ROUND_HALF_UP))


runs: 1,200 | test window: 2025-09-01 00:00:00+00:00 -> 2026-04-30 16:00:00+00:00 (5,801 hourly bars)


## Table 5.1 — Distributional summary of the sweep (`tab:sweep_overview`)

- **“Runs”** are the 1,200 seed-level trainings;
- **“Cells”** are the 240 seed-averaged configurations (one ablation cell under one algorithm and one target).

In [2]:
# "cells" = seed-averaged (cell, algorithm, target) configurations (240 of them)
cell_mean = summary.groupby(["cell_id", "algorithm", "target"])["test_sharpe_ratio"].mean()
n_runs, n_cells = len(summary), int(cell_mean.size)

t51 = pd.DataFrame({"Value": {
    "Total runs":                            f"{n_runs:,}",
    "Total cells (after seed aggregation)":  f"{n_cells}",
    "Runs with positive test total return":  n_pct(int((summary.test_total_return > 0).sum()), n_runs),
    "Runs with positive test Sharpe":        n_pct(int((summary.test_sharpe_ratio > 0).sum()), n_runs),
    "Runs with positive test Sortino":       n_pct(int((summary.test_sortino_ratio > 0).sum()), n_runs),
    "Cells with positive mean test Sharpe":  n_pct(int((cell_mean > 0).sum()), n_cells),
    "Cells with mean test Sharpe >= 0.5":    f"{int((cell_mean >= 0.5).sum())}",
    "Cells with mean test Sharpe >= 1.0":    f"{int((cell_mean >= 1.0).sum())}",
    "Best single-seed test Sharpe":          f"{summary.test_sharpe_ratio.max():.2f}",
    "Best single-seed test Sortino":         f"{summary.test_sortino_ratio.max():.2f}",
    "Best cell-level mean test Sharpe":      f"{cell_mean.max():.2f}",
    "Worst single-seed test Sharpe":         f"{summary.test_sharpe_ratio.min():.2f}",
    "Worst single-seed test Sortino":        f"{summary.test_sortino_ratio.min():.2f}",
    "Runs with at least one liquidation":    n_pct(int((summary.test_n_liquidations > 0).sum()), n_runs),
}})
t51.index.name = "Quantity"
display(t51)


,Value
Quantity,
Total runs,"1,200"
Total cells (after seed aggregation),240
Runs with positive test total return,442 (36.8%)
Runs with positive test Sharpe,469 (39.1%)
Runs with positive test Sortino,469 (39.1%)
Cells with positive mean test Sharpe,72 (30.0%)
Cells with mean test Sharpe >= 0.5,27
Cells with mean test Sharpe >= 1.0,4
Best single-seed test Sharpe,2.42


## Table 5.2 — Buy-and-hold baseline (`tab:baseline_test`)

- Passive long-only baseline over the test window, funding cash flows included (long positions pay positive funding rates, receive negative ones); no transaction costs charged.
- Per-asset baselines are 100% long; the portfolio baseline holds constant 25% weights.

In [3]:
def baseline_equity(weights: dict, funding: bool = True) -> pd.Series:
    """Constant-weight passive baseline: hourly growth 1 + sum_i w_i r_i, minus
    w_i * funding_rate at each settlement (the environment's cash-flow convention)."""
    w = pd.Series(weights, dtype=float).reindex(test_close.columns).fillna(0.0)
    rets = test_close.pct_change().fillna(0.0)
    step = 1.0 + rets.mul(w, axis=1).sum(axis=1)
    if funding:
        step = step - test_fund.mul(w, axis=1).sum(axis=1)
    step.iloc[0] = 1.0
    return step.cumprod()


def perf_metrics(equity: pd.Series) -> dict:
    """Same conventions as the experiment: ddof=0, sqrt(8760), drawdown on equity path."""
    r = equity.pct_change().dropna()
    mu, sd = r.mean(), r.std(ddof=0)
    dn = r[r < 0].std(ddof=0)
    return {
        "TR":      equity.iloc[-1] / equity.iloc[0] - 1.0,
        "AR":      (equity.iloc[-1] / equity.iloc[0]) ** (ANNUALIZATION / len(r)) - 1.0,
        "AnnVol":  sd * np.sqrt(ANNUALIZATION),
        "Sharpe":  mu / (sd + 1e-12) * np.sqrt(ANNUALIZATION),
        "Sortino": mu / (dn + 1e-12) * np.sqrt(ANNUALIZATION),
        "MDD":     float(-((equity - equity.cummax()) / equity.cummax()).min()),
    }


# matched baselines: 100% long per asset, 25% x 4 for the portfolio
base_stats = {a: perf_metrics(baseline_equity({a: 1.0})) for a in ASSETS}
base_stats["ALL"] = perf_metrics(baseline_equity({a: 0.25 for a in ASSETS}))

rows = {f"{a} (100% long)": base_stats[a] for a in ASSETS}
rows["Portfolio (25% × 4)"] = base_stats["ALL"]
t52 = pd.DataFrame(rows).T
t52 = pd.DataFrame({
    "TR":      t52["TR"].map("{:.3f}".format),
    "AR":      t52["AR"].map("{:.3f}".format),
    "AnnVol":  t52["AnnVol"].map("{:.3f}".format),
    "Sharpe":  t52["Sharpe"].map("{:.2f}".format),
    "Sortino": t52["Sortino"].map("{:.2f}".format),
    "MDD":     t52["MDD"].map("{:.3f}".format),
})
t52.index.name = "Baseline"
display(t52)


,TR,AR,AnnVol,Sharpe,Sortino,MDD
Baseline,,,,,,
BTCUSDT (100% long),-0.308,-0.426,0.453,-1.00,-1.28,0.508
ETHUSDT (100% long),-0.493,-0.641,0.645,-1.27,-1.63,0.623
SOLUSDT (100% long),-0.578,-0.728,0.718,-1.46,-1.90,0.697
XRPUSDT (100% long),-0.506,-0.655,0.654,-1.30,-1.76,0.637
Portfolio (25% × 4),-0.469,-0.616,0.575,-1.38,-1.78,0.600


## Tables 5.3 & 5.4 — Best cell per algorithm × architecture (`tab:headline_risk`, `tab:headline_behav`)

For each of the six algorithm × architecture combinations, the (ablation cell, target) pair with the highest mean test Sharpe across the five seeds. Values are mean ± std across seeds.

Cell codes carry one digit per ablation axis: **E** encoder (0 attention–LSTM, 1 flat extractor), **L** leverage (0 unlevered, 1 leveraged 3× with liquidation), **R** reward (0 differential Sharpe, 1 log return with turnover penalty), **S** episode (0 sequential, 1 random 720 h sub-episodes).

Behavioural columns (5.4): 
- **Turnover** — average fraction of equity re-traded per hour;
- **TradingCost** / **FundingCost** — total trading and funding costs paid over the test period;
- **Liq.** — share of runs whose test episode ended in a margin liquidation;
- **|w|** / **σ_w** — average size and variability of the positions held.

In [4]:
# seed-averaged cells; short names: tr/sh/so/dd = return/Sharpe/Sortino/drawdown,
# to/tc/fc/lq = turnover/trading cost/funding cost/liquidations, w/ws = |w|/sigma_w
cs = (summary.groupby(["algorithm", "arch", "cell_id", "target"])
       .agg(tr_m=("test_total_return", "mean"), tr_s=("test_total_return", "std"),
            sh_m=("test_sharpe_ratio", "mean"), sh_s=("test_sharpe_ratio", "std"),
            so_m=("test_sortino_ratio", "mean"), so_s=("test_sortino_ratio", "std"),
            dd_m=("test_max_drawdown", "mean"), dd_s=("test_max_drawdown", "std"),
            to=("test_mean_turnover", "mean"), tc=("test_total_trading_cost", "mean"),
            fc=("test_total_funding_cost", "mean"), lq=("test_n_liquidations", "mean"),
            w=("test_mean_abs_exposure", "mean"), ws=("test_std_exposure", "mean"))
       .reset_index())
# best cell = highest mean test Sharpe per algorithm x architecture
best = cs.loc[cs.groupby(["algorithm", "arch"])["sh_m"].idxmax()].set_index(["arch", "algorithm"])
order = [("portfolio", a) for a in ALGOS] + [("asset", a) for a in ALGOS]
best = best.loc[order]

t53 = pd.DataFrame({
    "Cell":    best["cell_id"],
    "Target":  best["target"].map(SHORT),
    "TR":      [pm(m, s, 3) for m, s in zip(best.tr_m, best.tr_s)],
    "Sharpe":  [pm(m, s) for m, s in zip(best.sh_m, best.sh_s)],
    "Sortino": [pm(m, s) for m, s in zip(best.so_m, best.so_s)],
})
display(t53)

t54 = pd.DataFrame({
    "Target":       best["target"].map(SHORT),
    "Turnover":     best["to"].map(f3),
    "TradingCost":  best["tc"].map(f3),
    "FundingCost":  best["fc"].map(f3),
    "Liq.":         best["lq"].map("{:.2f}".format),
    "|w|":          best["w"].map("{:.2f}".format),
    "σ_w":     best["ws"].map("{:.2f}".format),
})
display(t54)


Cell Target              TR       Sharpe      Sortino
arch      algorithm                                                              
portfolio PPO           E0L1R1S1    ALL   0.605 ± 0.489  0.96 ± 1.01  1.33 ± 1.41
          A2C           E0L0R1S0    ALL   0.207 ± 0.230  0.87 ± 1.07  1.22 ± 1.47
          RecurrentPPO  E1L1R1S0    ALL  -0.141 ± 0.389  0.02 ± 0.67  0.04 ± 0.93
asset     PPO           E1L1R0S0    XRP   0.820 ± 0.038  1.36 ± 0.02  1.82 ± 0.04
          A2C           E1L1R1S0    SOL   1.067 ± 0.000  1.49 ± 0.00  2.12 ± 0.00
          RecurrentPPO  E0L0R1S1    BTC   0.251 ± 0.101  0.97 ± 0.29  1.36 ± 0.41

Target Turnover TradingCost FundingCost  Liq.   |w|   σ_w
arch      algorithm                                                             
portfolio PPO             ALL    0.002       0.018       0.024  0.00  0.75  0.41
          A2C             ALL    0.000       0.001       0.005  0.00  0.25  0.22
          RecurrentPPO    ALL    0.095       0.429       0.005  0.00  0.75  0.67
asset     PPO             XRP    0.001       0.009       0.019  0.00  2.00  0.12
          A2C             SOL    0.000       0.002       0.085  0.00  2.00  0.00
          RecurrentPPO    BTC    0.001       0.005      -0.016  0.00  1.00  0.18

## Table 5.5 — DRL vs matched buy-and-hold baseline (`tab:rq2a`)

Each headline cell against its matched baseline: portfolio cells vs the equal-weight portfolio baseline, asset-specific cells vs their own asset's 100%-long baseline. For MDD, lower is better.

In [5]:
bsh = best["target"].map(lambda t: base_stats[t]["Sharpe"])
bso = best["target"].map(lambda t: base_stats[t]["Sortino"])
bdd = best["target"].map(lambda t: base_stats[t]["MDD"])

t55 = pd.DataFrame({
    "Target":       best["target"].map(SHORT),
    "DRL Sharpe":   [pm(m, s) for m, s in zip(best.sh_m, best.sh_s)],
    "Base Sharpe":  bsh.map("{:.2f}".format),
    "Δ Sharpe": [f"{m - b:+.2f}" for m, b in zip(best.sh_m, bsh)],
    "DRL Sortino":  [pm(m, s) for m, s in zip(best.so_m, best.so_s)],
    "Base Sortino": bso.map("{:.2f}".format),
    "Δ Sortino": [f"{m - b:+.2f}" for m, b in zip(best.so_m, bso)],
    "DRL MDD":      [pm(m, s) for m, s in zip(best.dd_m, best.dd_s)],
    "Base MDD":     bdd.map("{:.2f}".format),
})
display(t55)


Target   DRL Sharpe Base Sharpe Δ Sharpe  DRL Sortino Base Sortino Δ Sortino      DRL MDD Base MDD
arch      algorithm                                                                                                      
portfolio PPO             ALL  0.96 ± 1.01       -1.38    +2.33  1.33 ± 1.41        -1.78     +3.11  0.51 ± 0.15     0.60
          A2C             ALL  0.87 ± 1.07       -1.38    +2.24  1.22 ± 1.47        -1.78     +3.00  0.19 ± 0.07     0.60
          RecurrentPPO    ALL  0.02 ± 0.67       -1.38    +1.40  0.04 ± 0.93        -1.78     +1.82  0.48 ± 0.17     0.60
asset     PPO             XRP  1.36 ± 0.02       -1.30    +2.66  1.82 ± 0.04        -1.76     +3.58  0.57 ± 0.02     0.64
          A2C             SOL  1.49 ± 0.00       -1.46    +2.95  2.12 ± 0.00        -1.90     +4.02  0.51 ± 0.00     0.70
          RecurrentPPO    BTC  0.97 ± 0.29       -1.00    +1.97  1.36 ± 0.41        -1.28     +2.64  0.25 ± 0.00     0.51

## Tables 5.6 & 5.7 — Algorithm-marginal performance (`tab:algo_marginal_risk`, `tab:algo_marginal_behav`)

Each algorithm averaged over all of its 400 runs (16 cells × 5 targets × 5 seeds) — the *marginal* view: the typical run, in contrast with the best-cell view above. Behavioural columns as in Table 5.4.

In [6]:
# marginal view: each algorithm over all 400 of its runs
g = (summary.groupby("algorithm")
     .agg(n=("test_sharpe_ratio", "size"),
          tr_m=("test_total_return", "mean"), tr_s=("test_total_return", "std"),
          sh_m=("test_sharpe_ratio", "mean"), sh_s=("test_sharpe_ratio", "std"),
          so_m=("test_sortino_ratio", "mean"), so_s=("test_sortino_ratio", "std"),
          dd=("test_max_drawdown", "mean"),
          to=("test_mean_turnover", "mean"), tc=("test_total_trading_cost", "mean"),
          fc=("test_total_funding_cost", "mean"), lq=("test_n_liquidations", "mean"),
          w=("test_mean_abs_exposure", "mean"), ws=("test_std_exposure", "mean"))
     .loc[ALGOS])

t56 = pd.DataFrame({
    "n":       g["n"],
    "TR":      [pm(m, s) for m, s in zip(g.tr_m, g.tr_s)],
    "Sharpe":  [pm(m, s) for m, s in zip(g.sh_m, g.sh_s)],
    "Sortino": [pm(m, s) for m, s in zip(g.so_m, g.so_s)],
    "MDD":     g["dd"].map("{:.2f}".format),
})
display(t56)

t57 = pd.DataFrame({
    "Turnover":    g["to"].map(f3),
    "TradingCost": g["tc"].map(f3),
    "FundingCost": g["fc"].map(f3),
    "Liq.":        g["lq"].map(f3),
    "|w|":         g["w"].map("{:.2f}".format),
    "σ_w":    g["ws"].map("{:.2f}".format),
})
display(t57)


,n,TR,Sharpe,Sortino,MDD
algorithm,,,,,
PPO,400,-0.07 ± 0.62,-0.65 ± 2.52,-0.77 ± 3.21,0.57
A2C,400,-0.15 ± 0.61,-0.50 ± 1.55,-0.60 ± 2.04,0.58
RecurrentPPO,400,-0.27 ± 0.60,-1.38 ± 2.65,-1.70 ± 3.34,0.63


,Turnover,TradingCost,FundingCost,Liq.,|w|,σ_w
algorithm,,,,,,
PPO,0.051,0.160,0.005,0.028,1.28,0.38
A2C,0.021,0.047,0.008,0.033,1.30,0.18
RecurrentPPO,0.101,0.215,0.004,0.103,1.26,0.42


## Table 5.8 — Wilcoxon signed-rank tests, algorithm comparison (`tab:wilcoxon_h2b`)

Pairs are matched on identical cell and target conditions with seed-averaged test Sharpe (16 × 5 = 80 pairs per comparison); Δ is the first algorithm minus the second. H₀: zero median paired difference (two-sided, α = 0.05, Bonferroni threshold α/8 = 0.00625).

In [7]:
# seed-averaged Sharpe per (cell, target), one column per algorithm -> 80 matched pairs
sa = (summary.groupby(["cell_id", "target", "algorithm"])["test_sharpe_ratio"]
      .mean().unstack("algorithm"))

rows = []
for a, b in [("A2C", "PPO"), ("A2C", "RecurrentPPO"), ("PPO", "RecurrentPPO")]:
    d = (sa[a] - sa[b]).dropna()  # paired Sharpe differences
    stat, p = stats.wilcoxon(d)
    rows.append({
        "Comparison": f"{a} vs {b}",
        "n pairs": len(d),
        "Median ΔSharpe": f"{d.median():+.2f}",
        "p-value": f"{p:.3f}" if p >= 0.001 else f"{p:.4f}",
        "Decision": "Reject H0" if p < 0.05 else "Fail to reject H0",
    })
t58 = pd.DataFrame(rows).set_index("Comparison")
display(t58)


,n pairs,Median ΔSharpe,p-value,Decision
Comparison,,,,
A2C vs PPO,80,-0.03,0.233,Fail to reject H0
A2C vs RecurrentPPO,80,+0.40,0.0001,Reject H0
PPO vs RecurrentPPO,80,+0.53,0.0002,Reject H0


## Tables 5.9 & 5.10 — Architecture-marginal performance (`tab:arch_marginal_risk`, `tab:arch_marginal_behav`)

Portfolio = one agent over 4 assets (n = 240 runs); asset-specific = four agents, one per asset (n = 960 runs). Marginal means over those runs; behavioural columns as in Table 5.4.

In [8]:
# marginal view by architecture: 240 portfolio runs vs 960 asset-specific runs
ga = (summary.groupby("arch")
      .agg(n=("test_sharpe_ratio", "size"),
           tr_m=("test_total_return", "mean"), tr_s=("test_total_return", "std"),
           sh_m=("test_sharpe_ratio", "mean"), sh_s=("test_sharpe_ratio", "std"),
           so_m=("test_sortino_ratio", "mean"), so_s=("test_sortino_ratio", "std"),
           dd=("test_max_drawdown", "mean"),
           to=("test_mean_turnover", "mean"), tc=("test_total_trading_cost", "mean"),
           fc=("test_total_funding_cost", "mean"), lq=("test_n_liquidations", "mean"),
           w=("test_mean_abs_exposure", "mean"), ws=("test_std_exposure", "mean"))
      .loc[["portfolio", "asset"]]
      .rename(index={"portfolio": "Portfolio", "asset": "Asset-specific"}))

t59 = pd.DataFrame({
    "n":       ga["n"],
    "TR":      [pm(m, s) for m, s in zip(ga.tr_m, ga.tr_s)],
    "Sharpe":  [pm(m, s) for m, s in zip(ga.sh_m, ga.sh_s)],
    "Sortino": [pm(m, s) for m, s in zip(ga.so_m, ga.so_s)],
    "MDD":     ga["dd"].map("{:.2f}".format),
})
display(t59)

t510 = pd.DataFrame({
    "Turnover":    ga["to"].map(f3),
    "TradingCost": ga["tc"].map(f3),
    "FundingCost": ga["fc"].map(f3),
    "Liq.":        ga["lq"].map(f3),
    "|w|":         ga["w"].map("{:.2f}".format),
    "σ_w":    ga["ws"].map("{:.2f}".format),
})
display(t510)


,n,TR,Sharpe,Sortino,MDD
arch,,,,,
Portfolio,240,-0.31 ± 0.49,-2.26 ± 3.45,-2.84 ± 4.35,0.53
Asset-specific,960,-0.13 ± 0.64,-0.49 ± 1.77,-0.57 ± 2.28,0.61


,Turnover,TradingCost,FundingCost,Liq.,|w|,σ_w
arch,,,,,,
Portfolio,0.145,0.339,0.005,0.113,0.50,0.44
Asset-specific,0.036,0.091,0.006,0.040,1.47,0.30


## Table 5.11 — Wilcoxon signed-rank test, architecture comparison (`tab:wilcoxon_h2c`)

Each portfolio run is paired with the equal-weight mean of the four asset-specific runs sharing the same algorithm, cell, and seed (3 × 16 × 5 = 240 pairs); Δ is portfolio minus asset-specific.

In [9]:
# each portfolio run vs the mean of its four asset-specific counterparts (240 pairs)
port = (summary[summary.arch == "portfolio"]
        .set_index(["algorithm", "cell_id", "seed"])["test_sharpe_ratio"])
asset_mean = (summary[summary.arch == "asset"]
              .groupby(["algorithm", "cell_id", "seed"])["test_sharpe_ratio"].mean())
d_arch = (port - asset_mean).dropna()
stat, p = stats.wilcoxon(d_arch)
t511 = pd.DataFrame([{
    "Comparison": "Portfolio vs asset-specific (mean)",
    "n": len(d_arch),
    "Median Δ": f"{d_arch.median():+.2f}",
    "p-value": f"{p:.1e}",
    "Decision": "Reject H0" if p < 0.05 else "Fail to reject H0",
}]).set_index("Comparison")
display(t511)


,n,Median Δ,p-value,Decision
Comparison,,,,
Portfolio vs asset-specific (mean),240,-1.02,1.5e-11,Reject H0


## Tables 5.12–5.15 — The four ablation axes (`tab:axis_encoder`, `tab:axis_leverage`, `tab:axis_reward`, `tab:axis_episode`)

Each option averaged over its 600 runs (8 × 3 × 5 × 5); |Δ| is the absolute difference of the two option means. Option labels are the artifact's own values — LSTM / MLP (encoder), L=1 / L=3+liq (leverage), diff_sharpe / log_penalty (reward), random_subs / sequential (episode).

In [10]:
# each option averaged over its 600 runs
AXES = [("encoder", ["LSTM", "MLP"]),
        ("leverage", ["L=1", "L=3+liq"]),
        ("reward", ["diff_sharpe", "log_penalty"]),
        ("episode", ["random_subs", "sequential"])]

for axis, opt_order in AXES:
    gx = (summary.groupby(axis)
          .agg(sh_m=("test_sharpe_ratio", "mean"), sh_s=("test_sharpe_ratio", "std"),
               so_m=("test_sortino_ratio", "mean"), so_s=("test_sortino_ratio", "std"),
               tr_m=("test_total_return", "mean"), tr_s=("test_total_return", "std"),
               dd=("test_max_drawdown", "mean"),
               to=("test_mean_turnover", "mean"), lq=("test_n_liquidations", "mean"))
          .loc[opt_order])
    tab = pd.DataFrame({
        "Sharpe":   [pm(m, s) for m, s in zip(gx.sh_m, gx.sh_s)],
        "Sortino":  [pm(m, s) for m, s in zip(gx.so_m, gx.so_s)],
        "TR":       [pm(m, s) for m, s in zip(gx.tr_m, gx.tr_s)],
        "MDD":      gx["dd"].map("{:.2f}".format),
        "Turnover": gx["to"].map(f3),
        "Liq.":     gx["lq"].map(f3),
    }, index=gx.index)
    tab.loc["|Δ|"] = [f"{abs(gx.sh_m.iloc[0] - gx.sh_m.iloc[1]):.2f}",
                            f"{abs(gx.so_m.iloc[0] - gx.so_m.iloc[1]):.2f}", "", "", "", ""]
    tab.index.name = f"Option ({axis})"
    display(tab)


,Sharpe,Sortino,TR,MDD,Turnover,Liq.
Option (encoder),,,,,,
LSTM,-0.39 ± 1.67,-0.45 ± 2.18,-0.09 ± 0.61,0.57,0.029,0.027
MLP,-1.30 ± 2.76,-1.60 ± 3.48,-0.23 ± 0.62,0.62,0.086,0.082
|Δ|,0.91,1.15,,,,


,Sharpe,Sortino,TR,MDD,Turnover,Liq.
Option (leverage),,,,,,
L=1,-0.98 ± 2.32,-1.22 ± 2.99,-0.18 ± 0.48,0.49,0.034,0.000
L=3+liq,-0.70 ± 2.32,-0.83 ± 2.92,-0.15 ± 0.73,0.70,0.081,0.108
|Δ|,0.27,0.39,,,,


,Sharpe,Sortino,TR,MDD,Turnover,Liq.
Option (reward),,,,,,
diff_sharpe,-1.29 ± 2.69,-1.58 ± 3.38,-0.25 ± 0.60,0.62,0.094,0.090
log_penalty,-0.39 ± 1.78,-0.47 ± 2.35,-0.08 ± 0.63,0.57,0.021,0.018
|Δ|,0.90,1.12,,,,


,Sharpe,Sortino,TR,MDD,Turnover,Liq.
Option (episode),,,,,,
random_subs,-1.01 ± 2.32,-1.23 ± 2.90,-0.24 ± 0.60,0.61,0.049,0.050
sequential,-0.68 ± 2.32,-0.82 ± 3.00,-0.09 ± 0.63,0.57,0.066,0.058
|Δ|,0.33,0.41,,,,


## Table 5.16 — Wilcoxon signed-rank tests, ablation axes (`tab:wilcoxon_h2d`)

Each pair consists of two runs identical in every respect except the axis under test (600 pairs per axis); Δ is the first option minus the second.

In [11]:
AXIS_CONTRASTS = [
    ("Encoder",  "encoder",  "LSTM",        "MLP",         "LSTM vs flat MLP"),
    ("Reward",   "reward",   "log_penalty", "diff_sharpe", "log-penalty vs diff-Sharpe"),
    ("Episode",  "episode",  "sequential",  "random_subs", "sequential vs random 720h"),
    ("Leverage", "leverage", "L=3+liq",     "L=1",         "L=3+liq vs L=1"),
]
axis_cols = ["encoder", "leverage", "reward", "episode"]
rows = []
for name, axis, hi, lo, contrast in AXIS_CONTRASTS:
    others = [a for a in axis_cols if a != axis] + ["algorithm", "target", "seed"]
    # pivot pairs runs identical in everything but this axis -> 600 pairs
    piv = summary.pivot_table(index=others, columns=axis, values="test_sharpe_ratio")
    d = (piv[hi] - piv[lo]).dropna()
    stat, p = stats.wilcoxon(d)
    p_str = "<1e-12" if p < 1e-12 else ("<1e-9" if p < 1e-9 else f"{p:.1e}")
    rows.append({"Axis": name, "Contrast": contrast, "n": len(d),
                 "Mean Δ": f"{d.mean():+.2f}", "Med. Δ": f"{d.median():+.2f}",
                 "p-value": p_str,
                 "Decision": "Reject H0" if p < 0.05 else "Fail to reject H0"})
t516 = pd.DataFrame(rows).set_index("Axis")
display(t516)


,Contrast,n,Mean Δ,Med. Δ,p-value,Decision
Axis,,,,,,
Encoder,LSTM vs flat MLP,600,+0.91,+0.03,<1e-9,Reject H0
Reward,log-penalty vs diff-Sharpe,600,+0.90,+0.05,<1e-12,Reject H0
Episode,sequential vs random 720h,600,+0.33,+0.00,3.5e-04,Reject H0
Leverage,L=3+liq vs L=1,600,+0.27,+0.00,4.4e-04,Reject H0


## Tables A.1–A.3 — Appendix sweep listings (`tab:app_top30_cells`, `tab:app_top30_runs`, `tab:app_bottom30_runs`)

Top 30 seed-averaged cells by mean test Sharpe, and the 30 highest / 30 lowest single-seed runs.

In [12]:
# A.1 ranks the 240 seed-averaged cells; A.2/A.3 rank single-seed runs
cells_rank = cs.sort_values("sh_m", ascending=False).head(30).reset_index(drop=True)
tA1 = pd.DataFrame({
    "#":        range(1, 31),
    "Algorithm": cells_rank["algorithm"],
    "Arch.":     cells_rank["arch"],
    "Target":    cells_rank["target"].map(SHORT),
    "Cell":      cells_rank["cell_id"],
    "Sharpe":    [pm(m, s) for m, s in zip(cells_rank.sh_m, cells_rank.sh_s)],
    "Sortino":   [pm(m, s) for m, s in zip(cells_rank.so_m, cells_rank.so_s)],
    "TR":        cells_rank["tr_m"].map("{:+.3f}".format),
    "MDD":       cells_rank["dd_m"].map("{:.3f}".format),
    "Turn.":     cells_rank["to"].map("{:.3f}".format),
    "|w|":       cells_rank["w"].map("{:.2f}".format),
    "σ_w":  cells_rank["ws"].map("{:.2f}".format),
}).set_index("#")
display(tA1)


def runs_table(df30):
    return pd.DataFrame({
        "#":        range(1, len(df30) + 1),
        "Algorithm": df30["algorithm"].values,
        "Arch.":     df30["arch"].values,
        "Target":    df30["target"].map(SHORT).values,
        "Cell":      df30["cell_id"].values,
        "Seed":      df30["seed"].values,
        "Sharpe":    df30["test_sharpe_ratio"].map("{:.2f}".format).values,
        "Sortino":   df30["test_sortino_ratio"].map("{:.2f}".format).values,
        "TR":        df30["test_total_return"].map("{:+.3f}".format).values,
        "MDD":       df30["test_max_drawdown"].map("{:.3f}".format).values,
        "Turn.":     df30["test_mean_turnover"].map("{:.3f}".format).values,
        "|w|":       df30["test_mean_abs_exposure"].map("{:.2f}".format).values,
        "σ_w":  df30["test_std_exposure"].map("{:.2f}".format).values,
    }).set_index("#")


tA2 = runs_table(summary.nlargest(30, "test_sharpe_ratio"))
display(tA2)

tA3 = runs_table(summary.nsmallest(30, "test_sharpe_ratio"))
display(tA3)


,Algorithm,Arch.,Target,Cell,Sharpe,Sortino,TR,MDD,Turn.,|w|,σ_w
#,,,,,,,,,,,
1,A2C,asset,SOL,E1L1R1S0,1.49 ± 0.00,2.12 ± 0.00,+1.067,0.506,0.000,2.00,0.00
2,PPO,asset,XRP,E1L1R0S0,1.36 ± 0.02,1.82 ± 0.04,+0.820,0.574,0.001,2.00,0.12
3,PPO,asset,SOL,E1L1R0S0,1.23 ± 0.53,1.75 ± 0.75,+0.758,0.506,0.019,1.98,0.13
4,PPO,asset,BTC,E0L1R1S0,1.04 ± 0.10,1.45 ± 0.13,+0.415,0.463,0.003,2.00,0.10
5,RecurrentPPO,asset,BTC,E0L0R1S1,0.97 ± 0.29,1.36 ± 0.41,+0.251,0.250,0.001,1.00,0.18
6,PPO,portfolio,ALL,E0L1R1S1,0.96 ± 1.01,1.33 ± 1.41,+0.605,0.507,0.002,0.75,0.41
7,PPO,asset,SOL,E1L1R1S0,0.94 ± 1.28,1.37 ± 1.77,+0.726,0.585,0.008,1.99,0.54
8,A2C,portfolio,ALL,E0L0R1S0,0.87 ± 1.07,1.22 ± 1.47,+0.207,0.187,0.000,0.25,0.22
9,RecurrentPPO,asset,SOL,E1L1R1S0,0.81 ± 1.31,1.17 ± 1.79,+0.559,0.597,0.006,2.00,0.18


,Algorithm,Arch.,Target,Cell,Seed,Sharpe,Sortino,TR,MDD,Turn.,|w|,σ_w
#,,,,,,,,,,,,
1,PPO,asset,SOL,E0L0R1S0,10,2.42,3.46,+1.621,0.274,0.002,1.00,0.57
2,PPO,asset,SOL,E0L0R1S0,15,2.23,3.22,+1.393,0.367,0.012,0.99,0.56
3,RecurrentPPO,asset,ETH,E0L0R1S1,15,2.18,2.96,+1.177,0.284,0.003,1.00,1.00
4,RecurrentPPO,asset,SOL,E0L1R1S1,5,1.85,2.68,+1.894,0.424,0.039,1.99,1.64
5,PPO,asset,ETH,E0L1R1S1,5,1.71,2.41,+1.437,0.523,0.011,1.99,1.24
6,PPO,asset,SOL,E1L1R1S0,15,1.67,2.39,+1.447,0.508,0.009,1.99,0.42
7,RecurrentPPO,asset,SOL,E1L1R1S1,20,1.65,2.35,+1.384,0.481,0.004,2.00,0.18
8,PPO,asset,SOL,E0L1R1S0,10,1.63,2.35,+1.346,0.467,0.007,1.99,0.21
9,PPO,asset,ETH,E0L0R0S0,20,1.62,2.21,+0.725,0.316,0.001,1.00,0.15


,Algorithm,Arch.,Target,Cell,Seed,Sharpe,Sortino,TR,MDD,Turn.,|w|,σ_w
#,,,,,,,,,,,,
1,PPO,portfolio,ALL,E1L0R0S1,20,-17.98,-22.71,-0.906,0.906,0.416,0.25,0.28
2,RecurrentPPO,portfolio,ALL,E1L1R0S1,5,-12.44,-14.13,-0.935,0.935,1.267,0.73,0.85
3,PPO,portfolio,ALL,E1L0R0S1,10,-12.38,-15.11,-0.875,0.878,0.302,0.25,0.27
4,PPO,portfolio,ALL,E1L0R0S0,25,-12.02,-15.66,-0.840,0.840,0.291,0.25,0.26
5,PPO,portfolio,ALL,E1L0R0S0,15,-11.76,-14.28,-0.929,0.932,0.402,0.25,0.25
6,RecurrentPPO,portfolio,ALL,E1L1R0S0,15,-11.18,-13.10,-0.933,0.936,1.100,0.75,0.82
7,PPO,portfolio,ALL,E1L0R1S0,15,-11.09,-15.24,-0.818,0.821,0.324,0.25,0.29
8,RecurrentPPO,portfolio,ALL,E1L1R0S1,10,-11.04,-13.35,-0.933,0.939,0.998,0.74,0.84
9,RecurrentPPO,portfolio,ALL,E1L0R0S1,15,-10.84,-14.81,-0.789,0.791,0.257,0.25,0.26
